# IBM Data Analyst Capstone — Full Analysis Notebook

This notebook contains all data collection and processing work that feeds the capstone presentation.

## Sections

| Section | Slides | Description |
|---------|--------|-------------|
| **Part 1 — Collecting Job Data via API** | Slide 7 | Fetches job postings by technology and location |
| **Part 2 — Web Scraping: Popular Languages** | Slide 8 | Scrapes programming language salary data |
| **Part 3 — Survey Data Processing** | Slides 9–12, 14–16 | Generates top-10 CSVs for Cognos dashboards |

---


---

## Part 1 — Collecting Job Data Using APIs

**Presentation slide:** Job Postings (Slide 7)

This section uses a Jobs API to count open job postings by technology skill and by location. Results are saved to Excel files used in the presentation.


### 1.1 Imports and Setup

In [ ]:
import requests
import pandas as pd
import json
from openpyxl import Workbook

# Jobs API endpoint (IBM Skills Network dataset)
api_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/module%201/Accessing%20Data%20Using%20APIs/jobs.json"

print("Libraries loaded.")

### 1.2 Warm-Up — ISS Astronaut API

Before the main lab, a quick test with a public API to confirm requests are working.

In [ ]:
iss_url = "http://api.open-notify.org/astros.json"
response = requests.get(iss_url)

if response.ok:
    data = response.json()
    astronauts = data.get('people')
    print(f"There are {len(astronauts)} astronauts currently on the ISS:")
    for a in astronauts:
        print(f"  - {a.get('name')}")

### 1.3 Function: Count Jobs by Technology

In [ ]:
def get_number_of_jobs_T(technology):
    """
    Returns the count of job postings that mention a given technology
    in the Key Skills field.
    """
    response = requests.get(api_url)
    data = response.json()
    count = sum(
        technology.lower() in job.get("Key Skills", "").lower()
        for job in data
    )
    return technology, count

# Test
print(get_number_of_jobs_T("Python"))

### 1.4 Function: Count Jobs by Location

In [ ]:
def get_number_of_jobs_L(location):
    """
    Returns the count of job postings in a given location.
    """
    response = requests.get(api_url)
    data = response.json()
    count = sum(
        location.lower() in job.get("Location", "").lower()
        for job in data
    )
    return location, count

# Test
print(get_number_of_jobs_L("Los Angeles"))

### 1.5 Job Postings by Location → job-postings.xlsx

In [ ]:
locations = [
    "Los Angeles", "New York", "San Francisco",
    "Washington DC", "Seattle", "Austin", "Detroit"
]

wb = Workbook()
ws = wb.active
ws.append(["Location", "Number of Jobs"])

for location in locations:
    _, count = get_number_of_jobs_L(location)
    ws.append([location, count])
    print(f"{location}: {count}")

wb.save("job-postings.xlsx")
print("\nSaved: job-postings.xlsx")

### 1.6 Job Postings by Technology Skill → job-skills.xlsx

In [ ]:
skills = [
    "C", "C#", "C++", "Java", "JavaScript",
    "Python", "Scala", "Oracle",
    "SQL Server", "MySQL Server", "PostgreSQL", "MongoDB"
]

wb_skills = Workbook()
ws_skills = wb_skills.active
ws_skills.append(["Technology", "Number of Jobs"])

for skill in skills:
    _, count = get_number_of_jobs_T(skill)
    ws_skills.append([skill, count])
    print(f"{skill}: {count}")

wb_skills.save("job-skills.xlsx")
print("\nSaved: job-skills.xlsx")

---

## Part 2 — Web Scraping: Popular Programming Languages

**Presentation slide:** Popular Languages (Slide 8)

This section scrapes a table of programming languages and their average annual salaries from an IBM Skills Network page. Results are saved to `popular-languages.csv`.


### 2.1 Imports

In [ ]:
from bs4 import BeautifulSoup
import requests
import csv

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/datasets/Programming_Languages.html"
print("Libraries loaded.")

### 2.2 Download and Parse the Page

In [ ]:
data = requests.get(url).text
soup = BeautifulSoup(data, "html.parser")
print("Page downloaded and parsed.")

### 2.3 Scrape Language Name and Average Annual Salary

In [ ]:
table = soup.find("table")
rows = table.find_all("tr")[1:]  # skip header row

languages = []
for row in rows:
    cols = row.find_all("td")
    language = cols[1].text.strip()   # column 1: Language name
    salary   = cols[3].text.strip()   # column 3: Average Annual Salary
    languages.append({"Language": language, "Average Annual Salary": salary})

print(f"Scraped {len(languages)} languages:")
for l in languages:
    print(f"  {l['Language']}: {l['Average Annual Salary']}")

### 2.4 Save to popular-languages.csv

In [ ]:
with open("popular-languages.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Language", "Average Annual Salary"])
    writer.writeheader()
    writer.writerows(languages)

print("Saved: popular-languages.csv")

---

## Part 3 — Survey Data Processing

**Presentation slides:** Programming Language Trends (Slides 9–10), Database Trends (Slides 11–12), Dashboard Tabs 1 & 2 (Slides 14–15)

This section loads the Stack Overflow Developer Survey data and generates the 8 top-10 CSV files used as inputs for the IBM Cognos Analytics dashboards.

**Output files:**

| File | Dashboard Tab | Chart |
|------|--------------|-------|
| `tab_1_csv_language_top10.csv` | Current Technology Usage | Bar chart |
| `tab_1_csv_database_top10.csv` | Current Technology Usage | Column chart |
| `tab_1_csv_platform_top10.csv` | Current Technology Usage | Word cloud |
| `tab_1_csv_webframe_top10.csv` | Current Technology Usage | Bubble chart |
| `tab_2_csv_language_want_top10.csv` | Future Technology Trend | Bar chart |
| `tab_2_csv_database_want_top10.csv` | Future Technology Trend | Column chart |
| `tab_2_csv_platform_want_top10.csv` | Future Technology Trend | Tree map |
| `tab_2_csv_webframe_want_top10.csv` | Future Technology Trend | Bubble chart |


### 3.1 Imports and Load Survey Data

In [ ]:
import pandas as pd
import os

os.makedirs("exports", exist_ok=True)

df = pd.read_csv("survey_data_updated.csv")
print(f"Dataset shape: {df.shape}")
print(f"Total respondents: {len(df):,}")
df.head(2)

### 3.2 Helper Function

Survey responses use semicolons to separate multiple answers (e.g. `"Python;JavaScript;SQL"`). This function splits, counts, and returns the top N.

In [ ]:
def get_top_n(df, column, n=10, label=None):
    """
    Splits semicolon-separated survey responses, counts occurrences,
    and returns the top N as a DataFrame.

    Parameters:
        df     : source DataFrame
        column : column with semicolon-separated values
        n      : number of results to return (default 10)
        label  : output column name (defaults to column name)
    """
    if label is None:
        label = column
    counts = (
        df[column]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
        .head(n)
        .reset_index()
    )
    counts.columns = [label, "Count"]
    return counts

print("Helper function defined.")

### 3.3 Tab 1 — Current Technology Usage

#### Top 10 Languages — Have Worked With

In [ ]:
lang_have = get_top_n(df, "LanguageHaveWorkedWith", label="Language")
print(lang_have.to_string(index=False))
lang_have.to_csv("exports/tab_1_csv_language_top10.csv", index=False)
print("\nSaved: tab_1_csv_language_top10.csv")

#### Top 10 Databases — Have Worked With

In [ ]:
db_have = get_top_n(df, "DatabaseHaveWorkedWith", label="Database")
print(db_have.to_string(index=False))
db_have.to_csv("exports/tab_1_csv_database_top10.csv", index=False)
print("\nSaved: tab_1_csv_database_top10.csv")

#### Top 10 Platforms — Have Worked With

In [ ]:
platform_have = get_top_n(df, "PlatformHaveWorkedWith", label="Platform")
print(platform_have.to_string(index=False))
platform_have.to_csv("exports/tab_1_csv_platform_top10.csv", index=False)
print("\nSaved: tab_1_csv_platform_top10.csv")

#### Top 10 Web Frameworks — Have Worked With

In [ ]:
webframe_have = get_top_n(df, "WebframeHaveWorkedWith", label="Webframe")
print(webframe_have.to_string(index=False))
webframe_have.to_csv("exports/tab_1_csv_webframe_top10.csv", index=False)
print("\nSaved: tab_1_csv_webframe_top10.csv")

### 3.4 Tab 2 — Future Technology Trend

#### Top 10 Languages — Want to Work With

In [ ]:
lang_want = get_top_n(df, "LanguageWantToWorkWith", label="Language")
print(lang_want.to_string(index=False))
lang_want.to_csv("exports/tab_2_csv_language_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_language_want_top10.csv")

#### Top 10 Databases — Want to Work With

In [ ]:
db_want = get_top_n(df, "DatabaseWantToWorkWith", label="Database")
print(db_want.to_string(index=False))
db_want.to_csv("exports/tab_2_csv_database_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_database_want_top10.csv")

#### Top 10 Platforms — Want to Work With

In [ ]:
platform_want = get_top_n(df, "PlatformWantToWorkWith", label="Platform")
print(platform_want.to_string(index=False))
platform_want.to_csv("exports/tab_2_csv_platform_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_platform_want_top10.csv")

#### Top 10 Web Frameworks — Want to Work With

In [ ]:
webframe_want = get_top_n(df, "WebframeWantToWorkWith", label="Webframe")
print(webframe_want.to_string(index=False))
webframe_want.to_csv("exports/tab_2_csv_webframe_want_top10.csv", index=False)
print("\nSaved: tab_2_csv_webframe_want_top10.csv")

### 3.5 Summary

In [ ]:
files = [
    "tab_1_csv_language_top10.csv",
    "tab_1_csv_database_top10.csv",
    "tab_1_csv_platform_top10.csv",
    "tab_1_csv_webframe_top10.csv",
    "tab_2_csv_language_want_top10.csv",
    "tab_2_csv_database_want_top10.csv",
    "tab_2_csv_platform_want_top10.csv",
    "tab_2_csv_webframe_want_top10.csv",
]

print("All exports generated:")
for fname in files:
    path = f"exports/{fname}"
    rows = pd.read_csv(path).shape[0]
    print(f"  {fname} ({rows} rows)")

print("\nDone. Upload the exports/ folder to Cognos Analytics.")